# YOLO11n Stage 3.1 — Small/Hard-Priority + Localization Fine-Tuning

Standalone Colab notebook.

## Starting checkpoint
**YOLO11n Stage 3 `best.pt`** — same P3/P4/P5 architecture. No architecture change.

## Why Stage 3.1
Stage 3 is currently the strongest stable YOLO detector. This experiment targets the failure modes found by the validation and end-to-end audits:

- weak **Small Coral Recall**
- difficult **Sinularia / WavingHand** detection coverage
- **False Positives**
- **Localization near-misses**

## Intervention
**Stage3 best.pt → train-only targeted dataset → gentle fine-tuning**

Training dataset contains:

1. all original 452 train images
2. true-small-centered slices
3. Sinularia/WavingHand-priority slices
4. crowded-positive slices
5. a small number of other positive slices
6. **train-only hard negatives mined by Stage3 itself**

Validation remains the original **100 images / 354 GT boxes**.

## Important experimental rules

- No validation sample is used to construct training data.
- Detector test is not loaded or evaluated.
- YOLO remains **one detection class: `target_coral`**.
- Raw species labels are used **only for train-sample prioritization and validation diagnostics**.
- Small remains `<1%` of original image area.
- Crowded remains `>=4 GT objects/image`.
- No built-in Copy-Paste.
- MixUp/CutMix are OFF.
- No blur, downscale, JPEG degradation, or strong noise.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + ENVIRONMENT
# ============================================================

!pip -q install ultralytics==8.4.117 albumentations==2.0.8 pycocotools pyyaml

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict

import os
import json
import random
import shutil
import time
import math
import yaml

import numpy as np
import pandas as pd
import cv2
import torch
import albumentations as A
import ultralytics

from ultralytics import YOLO

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU runtime in Colab.")

DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("=" * 115)
print("ENVIRONMENT")
print("=" * 115)
print("Ultralytics    :", ultralytics.__version__)
print("Albumentations :", A.__version__)
print("PyTorch        :", torch.__version__)
print("GPU            :", GPU_NAME)
print(f"VRAM           : {VRAM_GB:.2f} GB")
print("STATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PROJECT PATHS + LOAD TRAIN/VAL SOURCES
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Coral_Reef_AI_Project"
)

DETECTION_V1 = (
    PROJECT_ROOT
    / "01_Data"
    / "Final"
    / "Detection_v1"
)

TRAIN_IMG_DIR = DETECTION_V1 / "images" / "train"
VAL_IMG_DIR = DETECTION_V1 / "images" / "validation"

TRAIN_COCO = (
    DETECTION_V1
    / "annotations"
    / "instances_train.json"
)

VAL_COCO = (
    DETECTION_V1
    / "annotations"
    / "instances_validation.json"
)

# Species metadata produced by the classification ROI pipeline.
# Used ONLY for prioritizing train slices and species diagnostics.
MANIFEST = (
    PROJECT_ROOT
    / "04_Dataset_Splits"
    / "classification_roi_manifest_split.csv"
)

STAGE3_BEST = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_TargetedSliced_SmallCrowded"
    / "Detection_Stage3"
    / "weights"
    / "best.pt"
)

EXP_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_1_SmallHardPriority_Localization"
)

EXP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

BUILD_ROOT = Path(
    "/content/Coral_YOLO11_Stage3_1_SmallHardPriority"
)

DATA_YAML = BUILD_ROOT / "data.yaml"


for p in [
    TRAIN_IMG_DIR,
    VAL_IMG_DIR,
    TRAIN_COCO,
    VAL_COCO,
    MANIFEST,
    STAGE3_BEST,
]:
    if not p.exists():
        raise FileNotFoundError(p)


with open(TRAIN_COCO, "r", encoding="utf-8") as f:
    train_coco = json.load(f)

with open(VAL_COCO, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

manifest_df = pd.read_csv(MANIFEST)

required_manifest_cols = {
    "roi_id",
    "image_stem",
    "raw_detection_label",
    "x1_px",
    "y1_px",
    "x2_px",
    "y2_px",
    "split",
}

missing_cols = required_manifest_cols - set(manifest_df.columns)

if missing_cols:
    raise RuntimeError(
        f"Manifest missing columns: {sorted(missing_cols)}"
    )

train_manifest = (
    manifest_df.loc[
        manifest_df["split"].astype(str) == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

val_manifest = (
    manifest_df.loc[
        manifest_df["split"].astype(str) == "validation"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(train_coco["images"]) == 452
assert len(train_coco["annotations"]) == 1540
assert len(val_coco["images"]) == 100
assert len(val_coco["annotations"]) == 354

assert len(train_manifest) == 1540
assert len(val_manifest) == 354

print("=" * 115)
print("SOURCE CHECK")
print("=" * 115)

print("Stage3 checkpoint:")
print(STAGE3_BEST)

print("\nDetection train     :", len(train_coco["images"]), "images /", len(train_coco["annotations"]), "boxes")
print("Detection validation:", len(val_coco["images"]), "images /", len(val_coco["annotations"]), "boxes")

print("\nSpecies metadata train:", len(train_manifest))
print("Species metadata val  :", len(val_manifest))

print("\nTrain species distribution:")
print(
    train_manifest["raw_detection_label"]
    .value_counts()
)

print("\nTEST SPLIT WAS NOT LOADED.")
print("STATUS: SOURCE_CHECK_PASSED")


In [ ]:
# ============================================================
# CELL 3 — BUILD UNIFIED TRAIN METADATA + SIZE/PRIORITY AUDIT
# ============================================================

SMALL_AREA_RATIO = 0.01
MEDIUM_AREA_RATIO = 0.10
CROWDED_GT_COUNT = 4

PRIORITY_SPECIES = {
    "Sinularia",
    "WavingHand",
}

# Map actual train image stems -> file paths.
train_path_by_stem = {
    p.stem: p
    for p in TRAIN_IMG_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
}

val_path_by_stem = {
    p.stem: p
    for p in VAL_IMG_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
}

assert len(train_path_by_stem) == 452
assert len(val_path_by_stem) == 100


# Image dimensions from COCO by stem.
train_coco_image_by_stem = {
    Path(im["file_name"]).stem: im
    for im in train_coco["images"]
}

val_coco_image_by_stem = {
    Path(im["file_name"]).stem: im
    for im in val_coco["images"]
}

if set(train_manifest["image_stem"].astype(str)) - set(train_path_by_stem):
    missing = sorted(
        set(train_manifest["image_stem"].astype(str))
        - set(train_path_by_stem)
    )
    raise RuntimeError(
        f"Train manifest references missing images: {missing[:10]}"
    )

if set(val_manifest["image_stem"].astype(str)) - set(val_path_by_stem):
    missing = sorted(
        set(val_manifest["image_stem"].astype(str))
        - set(val_path_by_stem)
    )
    raise RuntimeError(
        f"Validation manifest references missing images: {missing[:10]}"
    )


def add_geometry_features(df, coco_image_by_stem):
    out = df.copy()

    widths = []
    heights = []
    relative_areas = []
    buckets = []

    for _, r in out.iterrows():
        stem = str(r["image_stem"])
        meta = coco_image_by_stem[stem]

        W = float(meta["width"])
        H = float(meta["height"])

        bw = max(
            0.0,
            float(r["x2_px"]) - float(r["x1_px"])
        )

        bh = max(
            0.0,
            float(r["y2_px"]) - float(r["y1_px"])
        )

        rel = (
            bw * bh
            / max(W * H, 1.0)
        )

        if rel < SMALL_AREA_RATIO:
            bucket = "small"
        elif rel < MEDIUM_AREA_RATIO:
            bucket = "medium"
        else:
            bucket = "large"

        widths.append(W)
        heights.append(H)
        relative_areas.append(rel)
        buckets.append(bucket)

    out["image_width"] = widths
    out["image_height"] = heights
    out["relative_area"] = relative_areas
    out["size_bucket"] = buckets

    return out


train_meta = add_geometry_features(
    train_manifest,
    train_coco_image_by_stem,
)

val_meta = add_geometry_features(
    val_manifest,
    val_coco_image_by_stem,
)


train_gt_by_stem = {}

for stem, g in train_meta.groupby("image_stem"):
    rows = []

    for _, r in g.iterrows():
        rows.append({
            "roi_id": str(r["roi_id"]),
            "species": str(r["raw_detection_label"]),
            "bbox": np.array(
                [
                    float(r["x1_px"]),
                    float(r["y1_px"]),
                    float(r["x2_px"]),
                    float(r["y2_px"]),
                ],
                dtype=np.float32,
            ),
            "relative_area": float(r["relative_area"]),
            "size_bucket": str(r["size_bucket"]),
        })

    train_gt_by_stem[str(stem)] = rows


train_image_summary = (
    train_meta
    .groupby("image_stem")
    .agg(
        gt_count=("roi_id", "count"),
        small_gt_count=(
            "size_bucket",
            lambda x: int((x == "small").sum())
        ),
        priority_species_count=(
            "raw_detection_label",
            lambda x: int(
                x.astype(str).isin(PRIORITY_SPECIES).sum()
            )
        ),
    )
    .reset_index()
)

train_image_summary["crowded"] = (
    train_image_summary["gt_count"]
    >= CROWDED_GT_COUNT
)

train_image_summary["has_small"] = (
    train_image_summary["small_gt_count"]
    > 0
)

TRAIN_OBJECT_AUDIT = (
    EXP_ROOT
    / "stage31_train_object_priority_audit.csv"
)

TRAIN_IMAGE_AUDIT = (
    EXP_ROOT
    / "stage31_train_image_priority_audit.csv"
)

train_meta.to_csv(
    TRAIN_OBJECT_AUDIT,
    index=False,
)

train_image_summary.to_csv(
    TRAIN_IMAGE_AUDIT,
    index=False,
)


print("=" * 115)
print("TRAIN PRIORITY AUDIT")
print("=" * 115)

print("\nSize distribution:")
print(
    train_meta["size_bucket"]
    .value_counts()
    .reindex(["small", "medium", "large"])
    .fillna(0)
    .astype(int)
)

print("\nPriority species by size:")
print(
    pd.crosstab(
        train_meta["raw_detection_label"],
        train_meta["size_bucket"],
    )
    .reindex(
        ["Sinularia", "WavingHand"],
        fill_value=0,
    )
)

print("\nImages with small GT      :", int(train_image_summary["has_small"].sum()))
print("Crowded train images      :", int(train_image_summary["crowded"].sum()))
print(
    "Images with Sin/Wave GT  :",
    int((train_image_summary["priority_species_count"] > 0).sum())
)

print("\nSTATUS: PRIORITY_METADATA_READY")


In [ ]:
# ============================================================
# CELL 4 — TRAIN-ONLY HARD-NEGATIVE MINING WITH STAGE3
#
# Validation is NEVER used here.
#
# Hard-negative rule:
# - Stage3 predicts a relatively confident box on a TRAIN image
# - prediction IoU with every GT is < 0.05
# - candidate 640x640 crop contains virtually no annotated GT
#
# If there are not enough mined hard negatives, safe
# background grid patches are used as fallback.
# ============================================================

HARD_NEG_PRED_CONF = 0.30
HARD_NEG_MAX_IOU = 0.05

HARD_NEG_TILE = 640
MAX_HARD_NEGATIVES = 60
MAX_HARD_NEG_PER_IMAGE = 2

# Reject a negative crop if more than this fraction
# of ANY annotated GT falls inside the crop.
MAX_GT_VISIBLE_IN_NEGATIVE = 0.02


def box_iou_matrix(a, b):
    a = np.asarray(a, dtype=np.float32).reshape(-1, 4)
    b = np.asarray(b, dtype=np.float32).reshape(-1, 4)

    if len(a) == 0 or len(b) == 0:
        return np.zeros(
            (len(a), len(b)),
            dtype=np.float32,
        )

    lt = np.maximum(
        a[:, None, :2],
        b[None, :, :2],
    )

    rb = np.minimum(
        a[:, None, 2:],
        b[None, :, 2:],
    )

    wh = np.clip(
        rb - lt,
        0,
        None,
    )

    inter = wh[..., 0] * wh[..., 1]

    aa = (
        np.clip(a[:, 2] - a[:, 0], 0, None)
        * np.clip(a[:, 3] - a[:, 1], 0, None)
    )

    bb = (
        np.clip(b[:, 2] - b[:, 0], 0, None)
        * np.clip(b[:, 3] - b[:, 1], 0, None)
    )

    union = (
        aa[:, None]
        + bb[None, :]
        - inter
    )

    return inter / np.clip(
        union,
        1e-9,
        None,
    )


def centered_tile(
    cx,
    cy,
    W,
    H,
    tile_size,
):
    tw = min(int(tile_size), int(W))
    th = min(int(tile_size), int(H))

    x0 = int(round(float(cx) - tw / 2))
    y0 = int(round(float(cy) - th / 2))

    x0 = max(
        0,
        min(x0, max(int(W) - tw, 0))
    )

    y0 = max(
        0,
        min(y0, max(int(H) - th, 0))
    )

    return x0, y0, tw, th


def gt_visible_fraction_in_tile(
    gt_box,
    x0,
    y0,
    tw,
    th,
):
    gx1, gy1, gx2, gy2 = map(float, gt_box)

    ix1 = max(gx1, float(x0))
    iy1 = max(gy1, float(y0))
    ix2 = min(gx2, float(x0 + tw))
    iy2 = min(gy2, float(y0 + th))

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    gt_area = max(
        (gx2 - gx1) * (gy2 - gy1),
        1e-9,
    )

    return inter / gt_area


def tile_is_safe_negative(
    stem,
    x0,
    y0,
    tw,
    th,
):
    for gt in train_gt_by_stem.get(stem, []):
        frac = gt_visible_fraction_in_tile(
            gt["bbox"],
            x0,
            y0,
            tw,
            th,
        )

        if frac > MAX_GT_VISIBLE_IN_NEGATIVE:
            return False

    return True


stage3_for_mining = YOLO(
    str(STAGE3_BEST)
)

train_paths = [
    train_path_by_stem[stem]
    for stem in sorted(train_path_by_stem)
]

print("=" * 115)
print("MINING HARD NEGATIVES FROM TRAIN ONLY")
print("=" * 115)

mining_start = time.time()

mining_results = stage3_for_mining.predict(
    source=[str(p) for p in train_paths],
    imgsz=960,
    conf=HARD_NEG_PRED_CONF,
    iou=0.70,
    max_det=300,
    device=DEVICE,
    verbose=False,
)

print(
    f"Inference completed in "
    f"{(time.time() - mining_start) / 60:.2f} min"
)

hard_negative_candidates = []

for image_path, result in zip(
    train_paths,
    mining_results,
):
    stem = image_path.stem

    meta = train_coco_image_by_stem[stem]
    W = int(meta["width"])
    H = int(meta["height"])

    gt_boxes = np.asarray(
        [
            gt["bbox"]
            for gt in train_gt_by_stem.get(stem, [])
        ],
        dtype=np.float32,
    ).reshape(-1, 4)

    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):
        continue

    pred_boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    pred_conf = (
        result.boxes.conf
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    ious = box_iou_matrix(
        pred_boxes,
        gt_boxes,
    )

    for pred_idx, (box, conf) in enumerate(
        zip(pred_boxes, pred_conf)
    ):
        max_iou = (
            float(ious[pred_idx].max())
            if len(gt_boxes)
            else 0.0
        )

        if max_iou >= HARD_NEG_MAX_IOU:
            continue

        x1, y1, x2, y2 = map(float, box)

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0

        x0, y0, tw, th = centered_tile(
            cx,
            cy,
            W,
            H,
            HARD_NEG_TILE,
        )

        if not tile_is_safe_negative(
            stem,
            x0,
            y0,
            tw,
            th,
        ):
            continue

        hard_negative_candidates.append({
            "image_stem": stem,
            "source_path": str(image_path),
            "x0": x0,
            "y0": y0,
            "tile_w": tw,
            "tile_h": th,
            "det_conf": float(conf),
            "max_iou": max_iou,
            "source": "stage3_false_positive",
        })


# Deduplicate near-identical tile origins.
hn_unique = {}

for c in hard_negative_candidates:
    key = (
        c["image_stem"],
        c["x0"] // 32,
        c["y0"] // 32,
    )

    if (
        key not in hn_unique
        or c["det_conf"] > hn_unique[key]["det_conf"]
    ):
        hn_unique[key] = c

hard_negative_candidates = list(
    hn_unique.values()
)

# Highest-confidence model mistakes first,
# while limiting each source image.
hard_negative_candidates = sorted(
    hard_negative_candidates,
    key=lambda x: x["det_conf"],
    reverse=True,
)

selected_hard_negatives = []
per_image_hn = defaultdict(int)

for c in hard_negative_candidates:
    stem = c["image_stem"]

    if per_image_hn[stem] >= MAX_HARD_NEG_PER_IMAGE:
        continue

    selected_hard_negatives.append(c)
    per_image_hn[stem] += 1

    if len(selected_hard_negatives) >= MAX_HARD_NEGATIVES:
        break


# ------------------------------------------------------------
# Fallback: safe background grid tiles.
# Still TRAIN ONLY.
# ------------------------------------------------------------

def grid_starts(length, tile, overlap=0.20):
    if length <= tile:
        return [0]

    step = max(
        1,
        int(round(tile * (1.0 - overlap)))
    )

    starts = list(
        range(
            0,
            max(length - tile + 1, 1),
            step,
        )
    )

    last = length - tile

    if starts[-1] != last:
        starts.append(last)

    return sorted(set(starts))


if len(selected_hard_negatives) < MAX_HARD_NEGATIVES:
    fallback_candidates = []

    for stem in sorted(train_path_by_stem):
        meta = train_coco_image_by_stem[stem]

        W = int(meta["width"])
        H = int(meta["height"])

        tw = min(HARD_NEG_TILE, W)
        th = min(HARD_NEG_TILE, H)

        for y0 in grid_starts(H, th, 0.20):
            for x0 in grid_starts(W, tw, 0.20):
                if not tile_is_safe_negative(
                    stem,
                    x0,
                    y0,
                    tw,
                    th,
                ):
                    continue

                fallback_candidates.append({
                    "image_stem": stem,
                    "source_path": str(train_path_by_stem[stem]),
                    "x0": x0,
                    "y0": y0,
                    "tile_w": tw,
                    "tile_h": th,
                    "det_conf": 0.0,
                    "max_iou": 0.0,
                    "source": "safe_background_fallback",
                })

    rng_hn = random.Random(SEED)

    rng_hn.shuffle(
        fallback_candidates
    )

    existing_keys = {
        (
            c["image_stem"],
            c["x0"],
            c["y0"],
            c["tile_w"],
            c["tile_h"],
        )
        for c in selected_hard_negatives
    }

    for c in fallback_candidates:
        key = (
            c["image_stem"],
            c["x0"],
            c["y0"],
            c["tile_w"],
            c["tile_h"],
        )

        if key in existing_keys:
            continue

        selected_hard_negatives.append(c)
        existing_keys.add(key)

        if len(selected_hard_negatives) >= MAX_HARD_NEGATIVES:
            break


hard_negative_df = pd.DataFrame(
    selected_hard_negatives
)

HARD_NEG_CSV = (
    EXP_ROOT
    / "stage31_train_hard_negative_mining.csv"
)

hard_negative_df.to_csv(
    HARD_NEG_CSV,
    index=False,
)


print("\nCandidate Stage3 false-positive negatives:",
      len(hard_negative_candidates))

print("Selected total negatives:",
      len(selected_hard_negatives))

if len(hard_negative_df):
    print("\nSource breakdown:")
    print(
        hard_negative_df["source"]
        .value_counts()
    )

    mined = hard_negative_df[
        hard_negative_df["source"]
        == "stage3_false_positive"
    ]

    if len(mined):
        print(
            "\nMined FP confidence:"
        )
        print(
            mined["det_conf"]
            .describe()
        )

assert len(selected_hard_negatives) > 0

print("\nVALIDATION WAS NOT USED FOR MINING.")
print("STATUS: HARD_NEGATIVE_MINING_COMPLETE")


In [ ]:
# ============================================================
# CELL 5 — TARGETED POSITIVE SLICE CANDIDATES
#
# Priority order:
# 1) TRUE small GT
# 2) Sinularia/WavingHand small or medium GT
# 3) crowded positive tiles
# 4) limited other positives
#
# All generated from TRAIN only.
# ============================================================

SLICE_SIZE = 640
GRID_OVERLAP = 0.20
MIN_VISIBLE_FRACTION = 0.40

MAX_SMALL_SLICES = 320
MAX_PRIORITY_SPECIES_SLICES = 180
MAX_CROWDED_SLICES = 100
MAX_OTHER_POSITIVE_SLICES = 40

SMALL_JITTERS = [
    (0.00, 0.00),
    (0.12, 0.00),
    (-0.12, 0.00),
]

PRIORITY_SPECIES_JITTERS = [
    (0.00, 0.00),
    (0.08, 0.00),
]


def make_centered_origin(
    cx,
    cy,
    W,
    H,
    tile_size,
    jx=0.0,
    jy=0.0,
):
    tw = min(int(tile_size), int(W))
    th = min(int(tile_size), int(H))

    target_cx = float(cx) + float(jx) * tw
    target_cy = float(cy) + float(jy) * th

    x0 = int(round(target_cx - tw / 2))
    y0 = int(round(target_cy - th / 2))

    x0 = max(
        0,
        min(x0, max(int(W) - tw, 0))
    )

    y0 = max(
        0,
        min(y0, max(int(H) - th, 0))
    )

    return x0, y0, tw, th


def clip_gt_to_tile(
    gt,
    tile_x,
    tile_y,
    tile_w,
    tile_h,
):
    x1, y1, x2, y2 = map(
        float,
        gt["bbox"],
    )

    tx1 = float(tile_x)
    ty1 = float(tile_y)
    tx2 = float(tile_x + tile_w)
    ty2 = float(tile_y + tile_h)

    cx1 = max(x1, tx1)
    cy1 = max(y1, ty1)
    cx2 = min(x2, tx2)
    cy2 = min(y2, ty2)

    cw = max(0.0, cx2 - cx1)
    ch = max(0.0, cy2 - cy1)

    original_area = max(
        (x2 - x1) * (y2 - y1),
        1e-9,
    )

    visible_fraction = (
        cw * ch
        / original_area
    )

    if (
        cw <= 1.0
        or ch <= 1.0
        or visible_fraction < MIN_VISIBLE_FRACTION
    ):
        return None

    return {
        "roi_id": gt["roi_id"],
        "species": gt["species"],
        "x": cx1 - tx1,
        "y": cy1 - ty1,
        "w": cw,
        "h": ch,
        "visible_fraction": visible_fraction,
        "original_relative_area": gt["relative_area"],
        "original_size_bucket": gt["size_bucket"],
    }


def make_positive_candidate(
    stem,
    x0,
    y0,
    tw,
    th,
    reason,
):
    kept = []

    for gt in train_gt_by_stem[stem]:
        clipped = clip_gt_to_tile(
            gt,
            x0,
            y0,
            tw,
            th,
        )

        if clipped is not None:
            kept.append(clipped)

    return {
        "image_stem": stem,
        "source_path": str(train_path_by_stem[stem]),
        "x0": int(x0),
        "y0": int(y0),
        "tile_w": int(tw),
        "tile_h": int(th),
        "reason": reason,
        "annotations": kept,
        "positive": len(kept) > 0,
        "contains_small": any(
            a["original_size_bucket"] == "small"
            for a in kept
        ),
        "contains_priority_species": any(
            a["species"] in PRIORITY_SPECIES
            for a in kept
        ),
        "kept_box_count": len(kept),
        "crowded_source": (
            len(train_gt_by_stem[stem])
            >= CROWDED_GT_COUNT
        ),
    }


candidate_map = {}


def add_candidate(c):
    key = (
        c["image_stem"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )

    if key not in candidate_map:
        candidate_map[key] = c
        return

    old = candidate_map[key]

    old_reasons = set(
        old["reason"].split("+")
    )

    new_reasons = set(
        c["reason"].split("+")
    )

    old["reason"] = "+".join(
        sorted(
            old_reasons
            | new_reasons
        )
    )


# ------------------------------------------------------------
# A) TRUE SMALL-CENTERED
# ------------------------------------------------------------

for stem, gt_items in train_gt_by_stem.items():
    meta = train_coco_image_by_stem[stem]

    W = int(meta["width"])
    H = int(meta["height"])

    for gt in gt_items:
        if gt["size_bucket"] != "small":
            continue

        x1, y1, x2, y2 = map(
            float,
            gt["bbox"],
        )

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0

        for jx, jy in SMALL_JITTERS:
            x0, y0, tw, th = make_centered_origin(
                cx,
                cy,
                W,
                H,
                SLICE_SIZE,
                jx,
                jy,
            )

            add_candidate(
                make_positive_candidate(
                    stem,
                    x0,
                    y0,
                    tw,
                    th,
                    "small_centered",
                )
            )


# ------------------------------------------------------------
# B) SINULARIA / WAVINGHAND PRIORITY
#
# Focus on small+medium because these are the useful
# difficult-resolution cases.
# ------------------------------------------------------------

for stem, gt_items in train_gt_by_stem.items():
    meta = train_coco_image_by_stem[stem]

    W = int(meta["width"])
    H = int(meta["height"])

    for gt in gt_items:
        if gt["species"] not in PRIORITY_SPECIES:
            continue

        if gt["size_bucket"] not in {
            "small",
            "medium",
        }:
            continue

        x1, y1, x2, y2 = map(
            float,
            gt["bbox"],
        )

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0

        species_reason = (
            "priority_"
            + gt["species"]
        )

        for jx, jy in PRIORITY_SPECIES_JITTERS:
            x0, y0, tw, th = make_centered_origin(
                cx,
                cy,
                W,
                H,
                SLICE_SIZE,
                jx,
                jy,
            )

            add_candidate(
                make_positive_candidate(
                    stem,
                    x0,
                    y0,
                    tw,
                    th,
                    species_reason,
                )
            )


# ------------------------------------------------------------
# C) CROWDED GRID
# ------------------------------------------------------------

for stem, gt_items in train_gt_by_stem.items():
    if len(gt_items) < CROWDED_GT_COUNT:
        continue

    meta = train_coco_image_by_stem[stem]

    W = int(meta["width"])
    H = int(meta["height"])

    tw = min(SLICE_SIZE, W)
    th = min(SLICE_SIZE, H)

    for y0 in grid_starts(
        H,
        th,
        GRID_OVERLAP,
    ):
        for x0 in grid_starts(
            W,
            tw,
            GRID_OVERLAP,
        ):
            c = make_positive_candidate(
                stem,
                x0,
                y0,
                tw,
                th,
                "crowded_grid",
            )

            if (
                c["positive"]
                and c["kept_box_count"] >= 2
            ):
                add_candidate(c)


# ------------------------------------------------------------
# D) GENERAL POSITIVE GRID
# ------------------------------------------------------------

for stem in sorted(train_gt_by_stem):
    meta = train_coco_image_by_stem[stem]

    W = int(meta["width"])
    H = int(meta["height"])

    tw = min(SLICE_SIZE, W)
    th = min(SLICE_SIZE, H)

    for y0 in grid_starts(
        H,
        th,
        GRID_OVERLAP,
    ):
        for x0 in grid_starts(
            W,
            tw,
            GRID_OVERLAP,
        ):
            c = make_positive_candidate(
                stem,
                x0,
                y0,
                tw,
                th,
                "other_positive_grid",
            )

            if c["positive"]:
                add_candidate(c)


all_candidates = list(
    candidate_map.values()
)


small_candidates = [
    c
    for c in all_candidates
    if c["contains_small"]
]


priority_candidates = [
    c
    for c in all_candidates
    if (
        c["contains_priority_species"]
        and not c["contains_small"]
    )
]


crowded_candidates = [
    c
    for c in all_candidates
    if (
        c["crowded_source"]
        and c["positive"]
        and c["kept_box_count"] >= 2
        and not c["contains_small"]
        and not c["contains_priority_species"]
    )
]


other_candidates = [
    c
    for c in all_candidates
    if (
        c["positive"]
        and not c["contains_small"]
        and not c["contains_priority_species"]
        and not c["crowded_source"]
    )
]


rng = random.Random(SEED)


def sample_cap(items, n):
    items = list(items)

    if len(items) <= n:
        return items

    return rng.sample(
        items,
        n,
    )


chosen_small = sample_cap(
    small_candidates,
    MAX_SMALL_SLICES,
)

chosen_priority = sample_cap(
    priority_candidates,
    MAX_PRIORITY_SPECIES_SLICES,
)

chosen_crowded = sample_cap(
    crowded_candidates,
    MAX_CROWDED_SLICES,
)

chosen_other = sample_cap(
    other_candidates,
    MAX_OTHER_POSITIVE_SLICES,
)


selected_positive_candidates = (
    chosen_small
    + chosen_priority
    + chosen_crowded
    + chosen_other
)


# Final dedup.
selected_positive_unique = {}

for c in selected_positive_candidates:
    key = (
        c["image_stem"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )

    selected_positive_unique[key] = c

selected_positive_candidates = list(
    selected_positive_unique.values()
)


print("=" * 115)
print("TARGETED POSITIVE SLICE SELECTION")
print("=" * 115)

print("All positive candidates       :", len(all_candidates))
print("Small-containing available    :", len(small_candidates))
print("Priority-species available    :", len(priority_candidates))
print("Crowded available             :", len(crowded_candidates))
print("Other available               :", len(other_candidates))

print("\nSELECTED")
print("Small-priority                :", len(chosen_small))
print("Sin/Wave priority             :", len(chosen_priority))
print("Crowded                       :", len(chosen_crowded))
print("Other positive                :", len(chosen_other))
print("Final unique positive slices  :", len(selected_positive_candidates))

small_occurrences = sum(
    sum(
        a["original_size_bucket"] == "small"
        for a in c["annotations"]
    )
    for c in selected_positive_candidates
)

sin_occurrences = sum(
    sum(
        a["species"] == "Sinularia"
        for a in c["annotations"]
    )
    for c in selected_positive_candidates
)

wave_occurrences = sum(
    sum(
        a["species"] == "WavingHand"
        for a in c["annotations"]
    )
    for c in selected_positive_candidates
)

print("\nGT occurrences inside selected slices:")
print("Small      :", small_occurrences)
print("Sinularia  :", sin_occurrences)
print("WavingHand :", wave_occurrences)

assert len(selected_positive_candidates) > 0

print("\nSTATUS: TARGETED_POSITIVE_SLICES_READY")


In [ ]:
# ============================================================
# CELL 6 — MATERIALIZE STAGE3.1 YOLO DATASET
# ============================================================

if BUILD_ROOT.exists():
    shutil.rmtree(BUILD_ROOT)

for split in ["train", "val"]:
    (BUILD_ROOT / "images" / split).mkdir(
        parents=True,
        exist_ok=True,
    )

    (BUILD_ROOT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True,
    )


def safe_symlink(src, dst):
    src = Path(src)
    dst = Path(dst)

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    os.symlink(
        str(src),
        str(dst),
    )


def coco_original_to_yolo(
    coco,
    source_img_dir,
    out_img_dir,
    out_label_dir,
    prefix,
):
    anns_by_id = defaultdict(list)

    for ann in coco["annotations"]:
        anns_by_id[
            int(ann["image_id"])
        ].append(ann)

    image_count = 0
    box_count = 0

    for im in coco["images"]:
        image_id = int(im["id"])

        src_name = Path(
            im["file_name"]
        ).name

        src = (
            Path(source_img_dir)
            / src_name
        )

        if not src.exists():
            raise FileNotFoundError(src)

        dst_name = (
            f"{prefix}{src_name}"
        )

        safe_symlink(
            src,
            Path(out_img_dir)
            / dst_name,
        )

        W = float(im["width"])
        H = float(im["height"])

        lines = []

        for ann in anns_by_id[image_id]:
            x, y, bw, bh = map(
                float,
                ann["bbox"],
            )

            xc = (
                x + bw / 2.0
            ) / W

            yc = (
                y + bh / 2.0
            ) / H

            nw = bw / W
            nh = bh / H

            lines.append(
                f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
            )

            box_count += 1

        label_path = (
            Path(out_label_dir)
            / f"{Path(dst_name).stem}.txt"
        )

        label_path.write_text(
            "\n".join(lines)
            + ("\n" if lines else ""),
            encoding="utf-8",
        )

        image_count += 1

    return image_count, box_count


# Original train 452 / 1540.
orig_train_images, orig_train_boxes = coco_original_to_yolo(
    train_coco,
    TRAIN_IMG_DIR,
    BUILD_ROOT / "images" / "train",
    BUILD_ROOT / "labels" / "train",
    "orig__",
)

# Original validation 100 / 354.
val_images, val_boxes = coco_original_to_yolo(
    val_coco,
    VAL_IMG_DIR,
    BUILD_ROOT / "images" / "val",
    BUILD_ROOT / "labels" / "val",
    "val__",
)


slice_records = []
slice_box_count = 0


# ------------------------------------------------------------
# Positive slices
# ------------------------------------------------------------

for idx, c in enumerate(
    selected_positive_candidates
):
    src = Path(
        c["source_path"]
    )

    image = cv2.imread(
        str(src),
        cv2.IMREAD_COLOR,
    )

    if image is None:
        raise RuntimeError(
            f"Cannot read {src}"
        )

    x0 = c["x0"]
    y0 = c["y0"]
    tw = c["tile_w"]
    th = c["tile_h"]

    crop = image[
        y0:y0 + th,
        x0:x0 + tw,
    ]

    if crop.size == 0:
        raise RuntimeError(
            f"Empty positive crop: {c}"
        )

    slice_name = (
        f"pos_{idx:04d}"
        f"__{c['image_stem']}"
        f"__x{x0}_y{y0}.jpg"
    )

    out_img = (
        BUILD_ROOT
        / "images"
        / "train"
        / slice_name
    )

    ok = cv2.imwrite(
        str(out_img),
        crop,
        [
            int(cv2.IMWRITE_JPEG_QUALITY),
            95,
        ],
    )

    if not ok:
        raise RuntimeError(
            f"Failed writing {out_img}"
        )

    lines = []

    for a in c["annotations"]:
        xc = (
            a["x"] + a["w"] / 2.0
        ) / tw

        yc = (
            a["y"] + a["h"] / 2.0
        ) / th

        nw = a["w"] / tw
        nh = a["h"] / th

        lines.append(
            f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
        )

        slice_box_count += 1

    out_label = (
        BUILD_ROOT
        / "labels"
        / "train"
        / f"{Path(slice_name).stem}.txt"
    )

    out_label.write_text(
        "\n".join(lines)
        + ("\n" if lines else ""),
        encoding="utf-8",
    )

    slice_records.append({
        "sample_name": slice_name,
        "sample_type": "positive_slice",
        "image_stem": c["image_stem"],
        "x0": x0,
        "y0": y0,
        "tile_w": tw,
        "tile_h": th,
        "reason": c["reason"],
        "box_count": len(c["annotations"]),
        "contains_small": c["contains_small"],
        "contains_priority_species": c["contains_priority_species"],
    })


# ------------------------------------------------------------
# Hard negatives — empty labels
# ------------------------------------------------------------

for idx, c in enumerate(
    selected_hard_negatives
):
    src = Path(
        c["source_path"]
    )

    image = cv2.imread(
        str(src),
        cv2.IMREAD_COLOR,
    )

    if image is None:
        raise RuntimeError(
            f"Cannot read {src}"
        )

    x0 = int(c["x0"])
    y0 = int(c["y0"])
    tw = int(c["tile_w"])
    th = int(c["tile_h"])

    crop = image[
        y0:y0 + th,
        x0:x0 + tw,
    ]

    if crop.size == 0:
        raise RuntimeError(
            f"Empty negative crop: {c}"
        )

    neg_name = (
        f"hardneg_{idx:04d}"
        f"__{c['image_stem']}"
        f"__x{x0}_y{y0}.jpg"
    )

    out_img = (
        BUILD_ROOT
        / "images"
        / "train"
        / neg_name
    )

    ok = cv2.imwrite(
        str(out_img),
        crop,
        [
            int(cv2.IMWRITE_JPEG_QUALITY),
            95,
        ],
    )

    if not ok:
        raise RuntimeError(
            f"Failed writing {out_img}"
        )

    out_label = (
        BUILD_ROOT
        / "labels"
        / "train"
        / f"{Path(neg_name).stem}.txt"
    )

    out_label.write_text(
        "",
        encoding="utf-8",
    )

    slice_records.append({
        "sample_name": neg_name,
        "sample_type": "hard_negative",
        "image_stem": c["image_stem"],
        "x0": x0,
        "y0": y0,
        "tile_w": tw,
        "tile_h": th,
        "reason": c["source"],
        "box_count": 0,
        "contains_small": False,
        "contains_priority_species": False,
    })


dataset_manifest = pd.DataFrame(
    slice_records
)

DATASET_MANIFEST_CSV = (
    EXP_ROOT
    / "stage31_added_samples_manifest.csv"
)

dataset_manifest.to_csv(
    DATASET_MANIFEST_CSV,
    index=False,
)


data_yaml = {
    "path": str(BUILD_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "target_coral",
    },
}

with open(
    DATA_YAML,
    "w",
    encoding="utf-8",
) as f:
    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False,
    )


def count_yolo_boxes(label_dir):
    total = 0
    empty = 0
    invalid = []

    for lp in Path(label_dir).glob("*.txt"):
        txt = lp.read_text(
            encoding="utf-8"
        ).strip()

        if not txt:
            empty += 1
            continue

        for line in txt.splitlines():
            parts = line.split()

            if len(parts) != 5:
                invalid.append(
                    (lp.name, line)
                )
                continue

            cls, xc, yc, w, h = map(
                float,
                parts,
            )

            valid = (
                int(cls) == 0
                and 0 <= xc <= 1
                and 0 <= yc <= 1
                and 0 < w <= 1
                and 0 < h <= 1
            )

            if not valid:
                invalid.append(
                    (lp.name, line)
                )

            total += 1

    return total, empty, invalid


train_box_count, train_empty_labels, train_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "train"
)

val_box_count, val_empty_labels, val_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "val"
)


train_images_final = list(
    (BUILD_ROOT / "images" / "train")
    .glob("*")
)

val_images_final = list(
    (BUILD_ROOT / "images" / "val")
    .glob("*")
)


assert not train_invalid, train_invalid[:10]
assert not val_invalid, val_invalid[:10]

assert orig_train_images == 452
assert orig_train_boxes == 1540
assert val_images == 100
assert val_boxes == 354

assert train_box_count == (
    1540
    + slice_box_count
)

assert val_box_count == 354
assert len(val_images_final) == 100


print("=" * 115)
print("STAGE3.1 DATASET READY")
print("=" * 115)

print("Original train images :", orig_train_images)
print("Original train boxes  :", orig_train_boxes)

print("Positive slices       :", len(selected_positive_candidates))
print("Positive slice boxes  :", slice_box_count)

print("Hard negatives        :", len(selected_hard_negatives))
print("Empty train labels    :", train_empty_labels)

print("Final train images    :", len(train_images_final))
print("Final train boxes     :", train_box_count)

print("Validation images     :", len(val_images_final))
print("Validation boxes      :", val_box_count)

print("\nData YAML:")
print(DATA_YAML)

print("\nTEST IS NOT PRESENT.")
print("STATUS: STAGE31_DATASET_VALIDATED")


In [ ]:
# ============================================================
# CELL 7 — STAGE3.1 TRAINING
#
# Same YOLO11n P3/P4/P5 architecture.
# Gentle continuation from Stage3 best.pt.
# ============================================================

EPOCHS = 15
IMGSZ = 960
BATCH = 8 if VRAM_GB >= 12 else 6

LR0 = 5e-5
LRF = 0.10
WEIGHT_DECAY = 5e-4

BOX_WEIGHT = 9.0
DFL_WEIGHT = 1.75
CLS_WEIGHT = 0.5

RUN_NAME = (
    "Detection_Stage3_1_SmallHardPriority_Localization"
)


stage31_augmentations = [
    A.RGBShift(
        r_shift_limit=(-10, -2),
        g_shift_limit=(-2, 6),
        b_shift_limit=(2, 12),
        p=0.12,
    ),

    A.RandomBrightnessContrast(
        brightness_limit=(-0.10, 0.10),
        contrast_limit=(-0.12, 0.12),
        p=0.15,
    ),

    A.RandomGamma(
        gamma_limit=(88, 112),
        p=0.08,
    ),

    A.CLAHE(
        clip_limit=(1.0, 1.6),
        tile_grid_size=(8, 8),
        p=0.03,
    ),
]


model_stage31 = YOLO(
    str(STAGE3_BEST)
)

try:
    strides = [
        int(round(float(x)))
        for x in model_stage31.model.stride.detach().cpu().tolist()
    ]
except Exception:
    strides = None

print("=" * 120)
print("STAGE 3.1 TRAINING CONFIG")
print("=" * 120)

print("Start checkpoint :", STAGE3_BEST)
print("Architecture     : same Stage3 YOLO11n")
print("Strides          :", strides)
print("Epochs           :", EPOCHS)
print("Image size       :", IMGSZ)
print("Batch            :", BATCH)

print("\nOptimizer        : AdamW")
print("LR0              :", LR0)
print("LRF              :", LRF)

print("\nLoss:")
print("box              :", BOX_WEIGHT)
print("dfl              :", DFL_WEIGHT)
print("cls              :", CLS_WEIGHT)

print("\nAugmentation:")
print("mosaic           : 0.05")
print("close_mosaic     : final 5")
print("mixup            : OFF")
print("cutmix           : OFF")
print("copy_paste       : OFF")
print("multi_scale      : OFF")
print("blur/noise/jpeg  : OFF")

print("\nTEST             : NOT USED")


train_start = time.time()

results31 = model_stage31.train(
    data=str(DATA_YAML),

    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,

    device=DEVICE,
    workers=2,

    optimizer="AdamW",
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    cos_lr=True,
    warmup_epochs=0.5,

    multi_scale=0.0,

    degrees=4.0,
    translate=0.04,
    scale=0.10,
    shear=0.0,
    perspective=0.0,

    fliplr=0.50,
    flipud=0.05,

    mosaic=0.05,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    close_mosaic=5,

    hsv_h=0.004,
    hsv_s=0.10,
    hsv_v=0.08,

    augmentations=stage31_augmentations,

    box=BOX_WEIGHT,
    cls=CLS_WEIGHT,
    dfl=DFL_WEIGHT,

    amp=True,

    seed=SEED,
    deterministic=True,

    save=True,
    save_period=2,
    patience=100,
    plots=True,

    project=str(EXP_ROOT),
    name=RUN_NAME,
    exist_ok=True,

    cache=False,
    verbose=True,
)

training_minutes = (
    time.time()
    - train_start
) / 60.0

RUN_DIR = Path(
    results31.save_dir
)

WEIGHTS_DIR = (
    RUN_DIR
    / "weights"
)

BEST31_ULTRA = (
    WEIGHTS_DIR
    / "best.pt"
)

LAST31 = (
    WEIGHTS_DIR
    / "last.pt"
)

assert BEST31_ULTRA.exists()
assert LAST31.exists()


print("\n" + "=" * 115)
print("STAGE 3.1 TRAINING COMPLETE")
print("=" * 115)

print(
    f"Training time : "
    f"{training_minutes:.2f} min"
)

print("Run dir       :", RUN_DIR)
print("best.pt       :", BEST31_ULTRA)
print("last.pt       :", LAST31)

print("\nTEST WAS NOT EVALUATED.")
print("STATUS: STAGE31_TRAINING_COMPLETE")


In [ ]:
# ============================================================
# CELL 8 — VALIDATION AUDIT FUNCTIONS
#
# Measures:
# - confidence sweep
# - Precision / Recall / F1
# - Small / Medium / Large Recall
# - Crowded Recall
# - per-species detector recall
# - localization near-misses
# - pure misses
# ============================================================

val_gt_by_stem = {}

for stem, g in val_meta.groupby("image_stem"):
    rows = []

    for _, r in g.iterrows():
        rows.append({
            "roi_id": str(r["roi_id"]),
            "species": str(r["raw_detection_label"]),
            "bbox": np.array(
                [
                    float(r["x1_px"]),
                    float(r["y1_px"]),
                    float(r["x2_px"]),
                    float(r["y2_px"]),
                ],
                dtype=np.float32,
            ),
            "size_bucket": str(r["size_bucket"]),
        })

    val_gt_by_stem[str(stem)] = rows


original_val_paths = [
    val_path_by_stem[stem]
    for stem in sorted(val_path_by_stem)
]

assert len(original_val_paths) == 100


def greedy_match(
    pred_boxes,
    gt_boxes,
    threshold=0.50,
):
    pred_boxes = np.asarray(
        pred_boxes,
        dtype=np.float32,
    ).reshape(-1, 4)

    gt_boxes = np.asarray(
        gt_boxes,
        dtype=np.float32,
    ).reshape(-1, 4)

    if len(pred_boxes) == 0:
        return [], [], list(range(len(gt_boxes)))

    if len(gt_boxes) == 0:
        return [], list(range(len(pred_boxes))), []

    ious = box_iou_matrix(
        pred_boxes,
        gt_boxes,
    )

    candidates = []

    for p in range(len(pred_boxes)):
        for g in range(len(gt_boxes)):
            if ious[p, g] >= threshold:
                candidates.append(
                    (float(ious[p, g]), p, g)
                )

    candidates.sort(reverse=True)

    used_p = set()
    used_g = set()
    matches = []

    for iou, p, g in candidates:
        if p in used_p or g in used_g:
            continue

        used_p.add(p)
        used_g.add(g)

        matches.append(
            (p, g, iou)
        )

    unmatched_p = [
        p for p in range(len(pred_boxes))
        if p not in used_p
    ]

    unmatched_g = [
        g for g in range(len(gt_boxes))
        if g not in used_g
    ]

    return (
        matches,
        unmatched_p,
        unmatched_g,
    )


def prediction_dict(
    model,
    image_paths,
    conf=0.05,
):
    start = time.time()

    results = model.predict(
        source=[str(p) for p in image_paths],
        imgsz=960,
        conf=conf,
        iou=0.70,
        max_det=300,
        device=DEVICE,
        verbose=False,
    )

    wall_sec = (
        time.time()
        - start
    )

    pred_dict = {}
    inference_ms = []

    for image_path, result in zip(
        image_paths,
        results,
    ):
        if (
            result.boxes is None
            or len(result.boxes) == 0
        ):
            boxes = np.empty(
                (0, 4),
                dtype=np.float32,
            )

            confs = np.empty(
                (0,),
                dtype=np.float32,
            )

        else:
            boxes = (
                result.boxes.xyxy
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            confs = (
                result.boxes.conf
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

        pred_dict[
            image_path.stem
        ] = {
            "boxes": boxes,
            "conf": confs,
        }

        if (
            hasattr(result, "speed")
            and result.speed
            and "inference" in result.speed
        ):
            inference_ms.append(
                float(
                    result.speed["inference"]
                )
            )

    mean_model_ms = (
        float(np.mean(inference_ms))
        if inference_ms
        else None
    )

    wall_ms = (
        wall_sec
        / len(image_paths)
        * 1000.0
    )

    return (
        pred_dict,
        mean_model_ms,
        wall_ms,
    )


def threshold_sweep(pred_dict):
    rows = []

    for conf in np.round(
        np.arange(
            0.05,
            0.81,
            0.025,
        ),
        3,
    ):
        TP = FP = FN = 0

        for stem, gt_items in val_gt_by_stem.items():
            pred = pred_dict[stem]

            gt_boxes = np.asarray(
                [
                    x["bbox"]
                    for x in gt_items
                ],
                dtype=np.float32,
            ).reshape(-1, 4)

            keep = (
                pred["conf"]
                >= conf
            )

            pred_boxes = (
                pred["boxes"][keep]
            )

            matches, up, ug = greedy_match(
                pred_boxes,
                gt_boxes,
                0.50,
            )

            TP += len(matches)
            FP += len(up)
            FN += len(ug)

        precision = (
            TP / (TP + FP)
            if TP + FP
            else 0.0
        )

        recall = (
            TP / (TP + FN)
            if TP + FN
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall
            else 0.0
        )

        rows.append({
            "confidence": float(conf),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    df_sweep = pd.DataFrame(
        rows
    )

    best = df_sweep.loc[
        df_sweep["f1"].idxmax()
    ]

    return df_sweep, best


def detailed_audit(
    pred_dict,
    confidence,
):
    TP = FP = FN = 0

    gt_rows = []
    image_rows = []

    for stem, gt_items in val_gt_by_stem.items():
        pred = pred_dict[stem]

        gt_boxes = np.asarray(
            [
                x["bbox"]
                for x in gt_items
            ],
            dtype=np.float32,
        ).reshape(-1, 4)

        keep = (
            pred["conf"]
            >= confidence
        )

        pred_boxes = (
            pred["boxes"][keep]
        )

        pred_confs = (
            pred["conf"][keep]
        )

        matches, unmatched_pred, unmatched_gt = greedy_match(
            pred_boxes,
            gt_boxes,
            0.50,
        )

        TP += len(matches)
        FP += len(unmatched_pred)
        FN += len(unmatched_gt)

        matched_gt = {
            g: (p, iou)
            for p, g, iou in matches
        }

        all_ious = box_iou_matrix(
            pred_boxes,
            gt_boxes,
        )

        for g, gt in enumerate(gt_items):
            if g in matched_gt:
                p, iou = matched_gt[g]

                gt_rows.append({
                    "image_stem": stem,
                    "roi_id": gt["roi_id"],
                    "species": gt["species"],
                    "size": gt["size_bucket"],
                    "detected": True,
                    "iou": float(iou),
                    "det_conf": float(pred_confs[p]),
                    "failure_mode": "TP",
                })

            else:
                best_possible_iou = (
                    float(
                        all_ious[:, g].max()
                    )
                    if len(pred_boxes)
                    else 0.0
                )

                if (
                    best_possible_iou >= 0.10
                    and best_possible_iou < 0.50
                ):
                    failure_mode = (
                        "LOCALIZATION_ERROR"
                    )
                else:
                    failure_mode = (
                        "PURE_MISS"
                    )

                gt_rows.append({
                    "image_stem": stem,
                    "roi_id": gt["roi_id"],
                    "species": gt["species"],
                    "size": gt["size_bucket"],
                    "detected": False,
                    "iou": best_possible_iou,
                    "det_conf": np.nan,
                    "failure_mode": failure_mode,
                })

        image_rows.append({
            "image_stem": stem,
            "gt_count": len(gt_items),
            "crowded": (
                len(gt_items)
                >= CROWDED_GT_COUNT
            ),
            "TP": len(matches),
            "FP": len(unmatched_pred),
            "FN": len(unmatched_gt),
        })

    precision = (
        TP / (TP + FP)
        if TP + FP
        else 0.0
    )

    recall = (
        TP / (TP + FN)
        if TP + FN
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    gt_df = pd.DataFrame(
        gt_rows
    )

    image_df = pd.DataFrame(
        image_rows
    )

    size_rows = []

    for size in [
        "small",
        "medium",
        "large",
    ]:
        g = gt_df[
            gt_df["size"] == size
        ]

        size_rows.append({
            "size": size,
            "GT": len(g),
            "TP": int(g["detected"].sum()),
            "FN": int((~g["detected"]).sum()),
            "recall": (
                float(g["detected"].mean())
                if len(g)
                else np.nan
            ),
        })

    species_rows = []

    for species in sorted(
        gt_df["species"].unique()
    ):
        g = gt_df[
            gt_df["species"] == species
        ]

        species_rows.append({
            "species": species,
            "GT": len(g),
            "TP": int(g["detected"].sum()),
            "FN": int((~g["detected"]).sum()),
            "recall": (
                float(g["detected"].mean())
                if len(g)
                else np.nan
            ),
            "localization_errors": int(
                (
                    g["failure_mode"]
                    == "LOCALIZATION_ERROR"
                ).sum()
            ),
            "pure_misses": int(
                (
                    g["failure_mode"]
                    == "PURE_MISS"
                ).sum()
            ),
        })

    crowded_rows = []

    for crowded_value in [
        False,
        True,
    ]:
        g = image_df[
            image_df["crowded"]
            == crowded_value
        ]

        tp_c = int(
            g["TP"].sum()
        )

        fp_c = int(
            g["FP"].sum()
        )

        fn_c = int(
            g["FN"].sum()
        )

        p_c = (
            tp_c / (tp_c + fp_c)
            if tp_c + fp_c
            else 0.0
        )

        r_c = (
            tp_c / (tp_c + fn_c)
            if tp_c + fn_c
            else 0.0
        )

        crowded_rows.append({
            "crowded": crowded_value,
            "images": len(g),
            "TP": tp_c,
            "FP": fp_c,
            "FN": fn_c,
            "precision": p_c,
            "recall": r_c,
        })

    overall = {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "localization_errors": int(
            (
                gt_df["failure_mode"]
                == "LOCALIZATION_ERROR"
            ).sum()
        ),
        "pure_misses": int(
            (
                gt_df["failure_mode"]
                == "PURE_MISS"
            ).sum()
        ),
    }

    return (
        overall,
        pd.DataFrame(size_rows),
        pd.DataFrame(species_rows),
        pd.DataFrame(crowded_rows),
        gt_df,
        image_df,
    )


print("Validation GT:", sum(len(x) for x in val_gt_by_stem.values()))
print("Validation images:", len(original_val_paths))
print("STATUS: AUDIT_FUNCTIONS_READY")


In [ ]:
# ============================================================
# CELL 9 — AUDIT SAVED STAGE3.1 CHECKPOINTS
#
# We do not trust only Ultralytics best.pt.
# Every saved checkpoint is audited with project metrics.
# ============================================================

ANALYSIS_ROOT = (
    EXP_ROOT
    / "Checkpoint_Audit"
)

ANALYSIS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

candidate_ckpts = sorted(
    WEIGHTS_DIR.glob(
        "epoch*.pt"
    )
)

candidate_ckpts.extend([
    BEST31_ULTRA,
    LAST31,
])

unique_ckpts = []
seen = set()

for p in candidate_ckpts:
    if not p.exists():
        continue

    key = str(p.resolve())

    if key not in seen:
        unique_ckpts.append(p)
        seen.add(key)


print("=" * 115)
print("STAGE3.1 CHECKPOINTS TO AUDIT")
print("=" * 115)

for p in unique_ckpts:
    print(p.name)

print("Total:", len(unique_ckpts))


audit_rows = []
species_tables = {}


for checkpoint_path in unique_ckpts:
    print("\n" + "-" * 120)
    print("AUDITING:", checkpoint_path.name)
    print("-" * 120)

    model = YOLO(
        str(checkpoint_path)
    )

    ultra = model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=960,
        batch=BATCH,
        device=DEVICE,
        conf=0.001,
        iou=0.70,
        max_det=300,
        plots=False,
        verbose=False,
    )

    map50 = float(
        ultra.box.map50
    )

    map50_95 = float(
        ultra.box.map
    )

    pred_dict, model_ms, wall_ms = prediction_dict(
        model,
        original_val_paths,
        conf=0.05,
    )

    sweep_df, best_conf_row = threshold_sweep(
        pred_dict
    )

    best_conf = float(
        best_conf_row["confidence"]
    )

    (
        overall,
        size_df,
        species_df,
        crowded_df,
        gt_df,
        image_df,
    ) = detailed_audit(
        pred_dict,
        best_conf,
    )

    small_recall = float(
        size_df.loc[
            size_df["size"] == "small",
            "recall",
        ].iloc[0]
    )

    medium_recall = float(
        size_df.loc[
            size_df["size"] == "medium",
            "recall",
        ].iloc[0]
    )

    large_recall = float(
        size_df.loc[
            size_df["size"] == "large",
            "recall",
        ].iloc[0]
    )

    crowded_recall = float(
        crowded_df.loc[
            crowded_df["crowded"] == True,
            "recall",
        ].iloc[0]
    )

    sinularia_recall = float(
        species_df.loc[
            species_df["species"] == "Sinularia",
            "recall",
        ].iloc[0]
    )

    wavinghand_recall = float(
        species_df.loc[
            species_df["species"] == "WavingHand",
            "recall",
        ].iloc[0]
    )

    row = {
        "checkpoint": str(checkpoint_path),
        "checkpoint_name": checkpoint_path.name,

        "precision": float(
            overall["precision"]
        ),

        "overall_recall": float(
            overall["recall"]
        ),

        "f1": float(
            overall["f1"]
        ),

        "small_recall": small_recall,
        "medium_recall": medium_recall,
        "large_recall": large_recall,
        "crowded_recall": crowded_recall,

        "sinularia_recall": sinularia_recall,
        "wavinghand_recall": wavinghand_recall,

        "TP": int(overall["TP"]),
        "FP": int(overall["FP"]),
        "FN": int(overall["FN"]),

        "localization_errors": int(
            overall["localization_errors"]
        ),

        "pure_misses": int(
            overall["pure_misses"]
        ),

        "best_conf": best_conf,

        "map50": map50,
        "map50_95": map50_95,

        "model_inference_ms": model_ms,
        "wall_ms_per_image": float(wall_ms),
    }

    audit_rows.append(row)

    species_tables[
        checkpoint_path.name
    ] = species_df.copy()

    print(f"mAP50:95           : {map50_95:.4f}")
    print(f"mAP50              : {map50:.4f}")
    print(f"Best conf          : {best_conf:.3f}")

    print(f"Precision          : {row['precision']:.4f}")
    print(f"Overall Recall     : {row['overall_recall']:.4f}")
    print(f"F1                 : {row['f1']:.4f}")

    print(f"Small Recall       : {small_recall:.4f}")
    print(f"Medium Recall      : {medium_recall:.4f}")
    print(f"Large Recall       : {large_recall:.4f}")
    print(f"Crowded Recall     : {crowded_recall:.4f}")

    print(f"Sinularia Recall   : {sinularia_recall:.4f}")
    print(f"WavingHand Recall  : {wavinghand_recall:.4f}")

    print(
        "TP / FP / FN       :",
        row["TP"],
        "/",
        row["FP"],
        "/",
        row["FN"],
    )

    print(
        "Localization / Miss :",
        row["localization_errors"],
        "/",
        row["pure_misses"],
    )


stage31_audit_df = pd.DataFrame(
    audit_rows
)

AUDIT_CSV = (
    ANALYSIS_ROOT
    / "stage31_checkpoint_audit.csv"
)

stage31_audit_df.to_csv(
    AUDIT_CSV,
    index=False,
)

print("\nSTATUS: STAGE31_CHECKPOINT_AUDIT_COMPLETE")


In [ ]:
# ============================================================
# CELL 10 — SELECT BEST BALANCED CHECKPOINTS
#
# Save:
# A) best mAP50:95
# B) best operational F1
# C) best balanced small-object candidate
# ============================================================

best_map_row = stage31_audit_df.loc[
    stage31_audit_df["map50_95"].idxmax()
]

best_f1_row = stage31_audit_df.loc[
    stage31_audit_df["f1"].idxmax()
]


# Small-first candidate with safeguards:
# prefer highest Small Recall among checkpoints with
# Precision >= 0.75 and Large Recall >= 0.82.
safe_small_pool = stage31_audit_df[
    (stage31_audit_df["precision"] >= 0.75)
    & (stage31_audit_df["large_recall"] >= 0.82)
].copy()

if len(safe_small_pool) == 0:
    safe_small_pool = stage31_audit_df.copy()

best_small_row = (
    safe_small_pool
    .sort_values(
        by=[
            "small_recall",
            "overall_recall",
            "map50_95",
            "precision",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .iloc[0]
)


BEST_MAP_PATH = (
    ANALYSIS_ROOT
    / "best_stage31_map50_95.pt"
)

BEST_F1_PATH = (
    ANALYSIS_ROOT
    / "best_stage31_operational_f1.pt"
)

BEST_SMALL_PATH = (
    ANALYSIS_ROOT
    / "best_stage31_balanced_small.pt"
)


shutil.copy2(
    best_map_row["checkpoint"],
    BEST_MAP_PATH,
)

shutil.copy2(
    best_f1_row["checkpoint"],
    BEST_F1_PATH,
)

shutil.copy2(
    best_small_row["checkpoint"],
    BEST_SMALL_PATH,
)


print("=" * 120)
print("BEST STAGE3.1 CHECKPOINTS")
print("=" * 120)

print("\n[A] BEST mAP50:95")
print(
    best_map_row.to_string()
)

print("\n[B] BEST OPERATIONAL F1")
print(
    best_f1_row.to_string()
)

print("\n[C] BEST BALANCED SMALL")
print(
    best_small_row.to_string()
)

print("\nSaved:")
print(BEST_MAP_PATH)
print(BEST_F1_PATH)
print(BEST_SMALL_PATH)

print("\nSTATUS: STAGE31_BEST_CHECKPOINTS_SELECTED")


In [ ]:
# ============================================================
# CELL 11 — STAGE3 vs STAGE3.1 FINAL COMPARISON
# ============================================================

# Stage3 reference from the completed validation audit.
STAGE3_REF = {
    "method": "Stage3",
    "precision": 0.7937,
    "overall_recall": 0.7175,
    "f1": 0.7537,
    "small_recall": 0.4167,
    "medium_recall": 0.6981,
    "large_recall": 0.8585,
    "crowded_recall": 0.6822,
    "sinularia_recall": 0.5956,
    "wavinghand_recall": 0.6842,
    "TP": 254,
    "FP": 66,
    "FN": 100,
    "localization_errors": 32,
    "pure_misses": 68,
    "best_conf": 0.350,
    "map50": 0.7930,
    "map50_95": 0.4730,
}


def row_to_ref(row, method):
    return {
        "method": method,
        "precision": float(row["precision"]),
        "overall_recall": float(row["overall_recall"]),
        "f1": float(row["f1"]),
        "small_recall": float(row["small_recall"]),
        "medium_recall": float(row["medium_recall"]),
        "large_recall": float(row["large_recall"]),
        "crowded_recall": float(row["crowded_recall"]),
        "sinularia_recall": float(row["sinularia_recall"]),
        "wavinghand_recall": float(row["wavinghand_recall"]),
        "TP": int(row["TP"]),
        "FP": int(row["FP"]),
        "FN": int(row["FN"]),
        "localization_errors": int(row["localization_errors"]),
        "pure_misses": int(row["pure_misses"]),
        "best_conf": float(row["best_conf"]),
        "map50": float(row["map50"]),
        "map50_95": float(row["map50_95"]),
    }


stage31_map_ref = row_to_ref(
    best_map_row,
    "Stage3.1 Best mAP",
)

stage31_f1_ref = row_to_ref(
    best_f1_row,
    "Stage3.1 Best F1",
)

stage31_small_ref = row_to_ref(
    best_small_row,
    "Stage3.1 Best Balanced Small",
)


comparison_df = pd.DataFrame([
    STAGE3_REF,
    stage31_map_ref,
    stage31_f1_ref,
    stage31_small_ref,
])

COMPARISON_CSV = (
    ANALYSIS_ROOT
    / "stage3_vs_stage31_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_CSV,
    index=False,
)


print("=" * 165)
print("STAGE3 vs STAGE3.1 FINAL COMPARISON")
print("=" * 165)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "overall_recall": "{:.4f}".format,
            "f1": "{:.4f}".format,
            "small_recall": "{:.4f}".format,
            "medium_recall": "{:.4f}".format,
            "large_recall": "{:.4f}".format,
            "crowded_recall": "{:.4f}".format,
            "sinularia_recall": "{:.4f}".format,
            "wavinghand_recall": "{:.4f}".format,
            "best_conf": "{:.3f}".format,
            "map50": "{:.4f}".format,
            "map50_95": "{:.4f}".format,
        },
    )
)


print("\n" + "=" * 120)
print("STAGE3 → STAGE3.1 BEST-mAP DELTAS")
print("=" * 120)

for metric in [
    "precision",
    "overall_recall",
    "f1",
    "small_recall",
    "medium_recall",
    "large_recall",
    "crowded_recall",
    "sinularia_recall",
    "wavinghand_recall",
    "map50",
    "map50_95",
]:
    delta = (
        float(stage31_map_ref[metric])
        - float(STAGE3_REF[metric])
    )

    print(
        f"{metric:<22}: {delta:+.4f}"
    )

print(
    f"{'FP':<22}: "
    f"{stage31_map_ref['FP'] - STAGE3_REF['FP']:+d}"
)

print(
    f"{'FN':<22}: "
    f"{stage31_map_ref['FN'] - STAGE3_REF['FN']:+d}"
)

print(
    f"{'localization_errors':<22}: "
    f"{stage31_map_ref['localization_errors'] - STAGE3_REF['localization_errors']:+d}"
)

print(
    f"{'pure_misses':<22}: "
    f"{stage31_map_ref['pure_misses'] - STAGE3_REF['pure_misses']:+d}"
)


# ------------------------------------------------------------
# Success targets
# ------------------------------------------------------------

strong_candidates = stage31_audit_df[
    (stage31_audit_df["precision"] >= 0.78)
    & (stage31_audit_df["overall_recall"] >= 0.75)
    & (stage31_audit_df["small_recall"] >= 0.50)
    & (stage31_audit_df["crowded_recall"] >= 0.72)
    & (stage31_audit_df["map50_95"] >= 0.48)
    & (stage31_audit_df["FP"] <= 66)
    & (stage31_audit_df["FN"] < 90)
].copy()


print("\n" + "=" * 120)
print("STAGE3.1 SUCCESS CHECKS")
print("=" * 120)

print(
    "Any Precision >= 0.78      :",
    bool(
        (stage31_audit_df["precision"] >= 0.78).any()
    )
)

print(
    "Any Overall Recall >= 0.75 :",
    bool(
        (stage31_audit_df["overall_recall"] >= 0.75).any()
    )
)

print(
    "Any Small Recall >= 0.50   :",
    bool(
        (stage31_audit_df["small_recall"] >= 0.50).any()
    )
)

print(
    "Any Crowded Recall >= 0.72 :",
    bool(
        (stage31_audit_df["crowded_recall"] >= 0.72).any()
    )
)

print(
    "Any mAP50:95 >= 0.48       :",
    bool(
        (stage31_audit_df["map50_95"] >= 0.48).any()
    )
)

print(
    "Any FP <= Stage3 (66)      :",
    bool(
        (stage31_audit_df["FP"] <= 66).any()
    )
)

print(
    "Any Localization < Stage3  :",
    bool(
        (stage31_audit_df["localization_errors"] < 32).any()
    )
)

print(
    "Strong balanced candidates :",
    len(strong_candidates)
)


if len(strong_candidates):
    print("\nSTRONG BALANCED CANDIDATES")

    print(
        strong_candidates[
            [
                "checkpoint_name",
                "precision",
                "overall_recall",
                "f1",
                "small_recall",
                "crowded_recall",
                "sinularia_recall",
                "wavinghand_recall",
                "FP",
                "FN",
                "localization_errors",
                "map50_95",
                "best_conf",
            ]
        ]
        .sort_values(
            [
                "map50_95",
                "small_recall",
            ],
            ascending=False,
        )
        .to_string(index=False)
    )


print("\nSTATUS: STAGE3_VS_STAGE31_COMPARISON_COMPLETE")


In [ ]:
# ============================================================
# CELL 12 — SAVE FINAL EXPERIMENT SUMMARY
# ============================================================

def json_safe(obj):
    if isinstance(obj, dict):
        return {
            k: json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [
            json_safe(v)
            for v in obj
        ]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.bool_):
        return bool(obj)

    return obj


summary = {
    "experiment": (
        "YOLO11n_Stage3_1_SmallHardPriority_Localization"
    ),

    "starting_checkpoint": str(
        STAGE3_BEST
    ),

    "architecture": (
        "YOLO11n Stage3 unchanged P3/P4/P5"
    ),

    "training": {
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "optimizer": "AdamW",
        "lr0": LR0,
        "lrf": LRF,
        "box": BOX_WEIGHT,
        "dfl": DFL_WEIGHT,
        "cls": CLS_WEIGHT,
        "mosaic": 0.05,
        "close_mosaic": 5,
        "mixup": 0.0,
        "cutmix": 0.0,
        "copy_paste": 0.0,
        "multi_scale": 0.0,
    },

    "data": {
        "original_train_images": 452,
        "original_train_boxes": 1540,
        "positive_slices": int(
            len(selected_positive_candidates)
        ),
        "hard_negatives": int(
            len(selected_hard_negatives)
        ),
        "validation_images": 100,
        "validation_boxes": 354,
    },

    "best_map_checkpoint": str(
        BEST_MAP_PATH
    ),

    "best_f1_checkpoint": str(
        BEST_F1_PATH
    ),

    "best_balanced_small_checkpoint": str(
        BEST_SMALL_PATH
    ),

    "best_map_metrics": best_map_row.to_dict(),
    "best_f1_metrics": best_f1_row.to_dict(),
    "best_balanced_small_metrics": best_small_row.to_dict(),

    "strong_balanced_candidate_count": int(
        len(strong_candidates)
    ),

    "validation_used_for_training_data_generation": False,
    "test_used": False,
}


SUMMARY_JSON = (
    ANALYSIS_ROOT
    / "stage31_final_summary.json"
)

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        json_safe(summary),
        f,
        indent=2,
        ensure_ascii=False,
    )


print("=" * 120)
print("STAGE 3.1 COMPLETE")
print("=" * 120)

print("\nBest mAP checkpoint:")
print(BEST_MAP_PATH)

print("\nBest operational F1 checkpoint:")
print(BEST_F1_PATH)

print("\nBest balanced Small checkpoint:")
print(BEST_SMALL_PATH)

print("\nAudit CSV:")
print(AUDIT_CSV)

print("\nComparison CSV:")
print(COMPARISON_CSV)

print("\nDataset manifest:")
print(DATASET_MANIFEST_CSV)

print("\nHard-negative mining CSV:")
print(HARD_NEG_CSV)

print("\nSummary JSON:")
print(SUMMARY_JSON)

print("\nTEST SET WAS NOT USED.")
print("STATUS: STAGE3_1_COMPLETE")


## After Stage 3.1 finishes

Send the outputs from:

- `STAGE3.1 CHECKPOINTS TO AUDIT`
- each checkpoint audit
- `STAGE3 vs STAGE3.1 FINAL COMPARISON`
- `STAGE3.1 SUCCESS CHECKS`

The important decision metrics are:

- Precision
- Overall Recall
- Small Recall
- Crowded Recall
- Sinularia Recall
- WavingHand Recall
- FP / FN
- Localization Errors
- mAP50:95

If Stage 3.1 wins, use its selected checkpoint in the existing **Stage3 + MobileNetV4 V3 End-to-End notebook** by changing only `DETECTOR_CKPT`.
